# 05. LLM 식단 가이드 출력

**목적**: RAG 검색 결과 + 영양소 기준을 GPT-4o-mini에 전달하여 식단 가이드 생성

**입력**: 사용자 건강 수치, 영양소 기준, RAG 검색 결과  
**출력**: 아침/점심/저녁 식단, 권장/제한 식품, 영양소 요약

## 0. 패키지 설치 및 라이브러리 로드

In [ ]:
!pip install -q langchain langchain-community langchain-openai chromadb pymupdf openai

In [ ]:
import os
import warnings
from google.colab import drive, userdata

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.vectorstores import Chroma

warnings.filterwarnings('ignore')

drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/AH_03_06'

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

## 1. 벡터 DB 및 모델 로드

In [ ]:
# 벡터 DB 로드
CHROMA_PATH = f'{BASE}/data/chroma_db'
embeddings  = OpenAIEmbeddings(model='text-embedding-3-small')
vectordb    = Chroma(persist_directory=CHROMA_PATH, embedding_function=embeddings)

# GPT-4o-mini
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.3)

print('벡터 DB 및 LLM 로드 완료')

## 2. 영양소 기준 산출 함수 로드

In [ ]:
GROUP_TO_MEAL_PLAN = {
    '정상군':           'Balanced Diet',
    '고혈압군':          'Low-Sodium Diet',
    '혈당이상군':         'Low-Carb Diet',
    '비만군':            'Low-Calorie Diet',
    '고혈압+혈당이상군':   'Low-Carb Low-Sodium Diet',
    '고혈압+비만군':      'Low-Calorie Low-Sodium Diet',
    '혈당이상+비만군':    'Low-Carb Low-Calorie Diet',
    '복합위험군':         'Therapeutic Diet',
}

GROUP_COEFFICIENTS = {
    '정상군':           {'kcal_per_kg': 30, 'protein_per_kg': 0.9, 'carb_ratio': 0.55, 'fat_ratio': 0.25},
    '고혈압군':          {'kcal_per_kg': 30, 'protein_per_kg': 0.9, 'carb_ratio': 0.55, 'fat_ratio': 0.25},
    '혈당이상군':         {'kcal_per_kg': 28, 'protein_per_kg': 1.0, 'carb_ratio': 0.45, 'fat_ratio': 0.25},
    '비만군':            {'kcal_per_kg': 25, 'protein_per_kg': 1.0, 'carb_ratio': 0.50, 'fat_ratio': 0.25},
    '고혈압+혈당이상군':   {'kcal_per_kg': 28, 'protein_per_kg': 1.0, 'carb_ratio': 0.45, 'fat_ratio': 0.25},
    '고혈압+비만군':      {'kcal_per_kg': 25, 'protein_per_kg': 1.0, 'carb_ratio': 0.50, 'fat_ratio': 0.20},
    '혈당이상+비만군':    {'kcal_per_kg': 25, 'protein_per_kg': 1.1, 'carb_ratio': 0.45, 'fat_ratio': 0.20},
    '복합위험군':         {'kcal_per_kg': 25, 'protein_per_kg': 1.1, 'carb_ratio': 0.45, 'fat_ratio': 0.20},
}

GROUP_SEARCH_KEYWORDS = {
    '정상군':           '성인 균형 식단 권장 영양소',
    '고혈압군':          '고혈압 나트륨 제한 식사요법',
    '혈당이상군':         '당뇨 전단계 탄수화물 제한 식사요법',
    '비만군':            '비만 칼로리 제한 체중 감량 식단',
    '고혈압+혈당이상군':   '고혈압 당뇨 복합 식사요법 나트륨 탄수화물',
    '고혈압+비만군':      '고혈압 비만 칼로리 나트륨 제한',
    '혈당이상+비만군':    '당뇨 비만 저탄수화물 저칼로리 식단',
    '복합위험군':         '당뇨 고혈압 비만 복합 치료 식이요법',
}

def calculate_nutrient(age, gender, height, weight, group):
    coef     = GROUP_COEFFICIENTS.get(group, GROUP_COEFFICIENTS['정상군'])
    calories = round(weight * coef['kcal_per_kg'])
    if age >= 65:    calories = round(calories * 0.9)
    if gender == 2:  calories = round(calories * 0.9)
    protein  = round(weight * coef['protein_per_kg'], 1)
    carbs    = round(calories * coef['carb_ratio'] / 4, 1)
    fats     = round(calories * coef['fat_ratio'] / 9, 1)
    return {
        'group':     group,
        'meal_plan': GROUP_TO_MEAL_PLAN.get(group, 'Balanced Diet'),
        'calories':  calories,
        'protein':   protein,
        'carbs':     carbs,
        'fats':      fats,
    }

def assign_group(sbp, dbp, fbs, bmi, waist=None, gender=1):
    sbp_c   = 0 if sbp < 120 else (1 if sbp < 140 else 2)
    dbp_c   = 0 if dbp < 80  else (1 if dbp < 90  else 2)
    fbs_c   = 0 if fbs < 100 else (1 if fbs < 126 else 2)
    bmi_c   = 0 if bmi < 23  else (1 if bmi < 25  else 2)
    waist_c = 0 if waist is None else (2 if (waist >= 85 if gender == 2 else waist >= 90) else 0)

    hypertension = (sbp_c >= 1) or (dbp_c >= 1)
    glucose      = (fbs_c >= 1)
    obesity      = (bmi_c >= 1) or (waist_c >= 1)
    count = sum([hypertension, glucose, obesity])

    if count == 0:                       return '정상군'
    elif count == 3:                     return '복합위험군'
    elif hypertension and glucose:       return '고혈압+혈당이상군'
    elif hypertension and obesity:       return '고혈압+비만군'
    elif glucose and obesity:            return '혈당이상+비만군'
    elif hypertension:                   return '고혈압군'
    elif glucose:                        return '혈당이상군'
    else:                                return '비만군'

print('함수 로드 완료')

## 3. 식단 가이드 생성 함수

In [ ]:
def generate_diet_guide(age, gender, height, weight, sbp, dbp, fbs, waist=None):
    """
    사용자 건강 수치를 입력받아 식단 가이드를 생성합니다.
    """
    # 1. 그룹 결정
    bmi   = round(weight / ((height / 100) ** 2), 1)
    group = assign_group(sbp, dbp, fbs, bmi, waist, gender)

    # 2. 영양소 기준 산출
    nutrient = calculate_nutrient(age, gender, height, weight, group)

    # 3. RAG 검색
    keyword  = GROUP_SEARCH_KEYWORDS.get(group, '균형 식단')
    rag_docs = vectordb.similarity_search(keyword, k=3)
    rag_text = '\n'.join([f'- ({doc.metadata["source"]}) {doc.page_content[:150]}' for doc in rag_docs])

    # 4. 프롬프트 구성
    prompt = f"""당신은 한국인 영양사입니다. 아래 정보를 바탕으로 개인 맞춤형 식단 가이드를 작성해주세요.

[사용자 정보]
나이: {age}세 / 성별: {'남성' if gender == 1 else '여성'}
신장: {height}cm / 체중: {weight}kg / BMI: {bmi}
수축기혈압: {sbp}mmHg / 이완기혈압: {dbp}mmHg / 공복혈당: {fbs}mg/dL
건강 그룹: {group}

[영양소 기준]
식단 플랜: {nutrient['meal_plan']}
권장 칼로리: {nutrient['calories']} kcal
권장 단백질: {nutrient['protein']} g
권장 탄수화물: {nutrient['carbs']} g
권장 지방: {nutrient['fats']} g

[의학 근거]
{rag_text}

위 정보를 바탕으로 다음 형식으로 작성해주세요.

[식단 가이드]
아침:
점심:
저녁:

[권장 식품]

[제한 식품]

[영양소 요약]
칼로리 / 탄수화물 / 단백질 / 지방
"""

    # 5. LLM 호출
    response = llm.invoke(prompt)

    print(f'\n그룹: {group}')
    print(f'식단 플랜: {nutrient["meal_plan"]}')
    print(f'칼로리: {nutrient["calories"]} kcal / 단백질: {nutrient["protein"]}g / 탄수화물: {nutrient["carbs"]}g / 지방: {nutrient["fats"]}g')
    print('\n' + '='*60)
    print(response.content)

    return response.content

print('식단 가이드 생성 함수 정의 완료')

## 4. 테스트

In [ ]:
print('=== 테스트 1: 45세 남성 혈당이상 ===')
generate_diet_guide(
    age=45, gender=1, height=175, weight=80,
    sbp=118, dbp=76, fbs=115
)

In [ ]:
print('=== 테스트 2: 55세 여성 고혈압+비만 ===')
generate_diet_guide(
    age=55, gender=2, height=160, weight=75,
    sbp=145, dbp=92, fbs=95, waist=88
)

In [ ]:
print('=== 테스트 3: 62세 남성 복합위험군 ===')
generate_diet_guide(
    age=62, gender=1, height=168, weight=85,
    sbp=150, dbp=95, fbs=130, waist=96
)